# Semantic Scholar Batch PDF OCR Pipeline (protokol-7)

This notebook runs **Stage 2 & 3 OCR** on Semantic Scholar open-access research papers harvested by the `semanticscholar_pipeline`.

### Supported Hardware & Models:
- **Colab Free (T4 GPU - 16 GB VRAM)**: Uses `Qwen/Qwen2.5-VL-7B-Instruct` (4-bit NF4 quantized).
- **Colab Pro / Enterprise (A100 GPU - 40 GB+ VRAM)**: Uses `baidu/Unlimited-OCR` (pinned commit `07dea832e22aefee32ad281d4b80551282e1c168`).

### Pipeline Mechanism:
1. Reads Zstd-compressed Parquet shards from Google Drive where `pdf_ocr_needed == 1`.
2. Rasterizes PDF pages into images using PyMuPDF / pdfium.
3. Transcribes document layouts, text, headings, and tables into Markdown.
4. Updates Parquet shards (`pdf_text`, `pdf_ocr_needed = 0`, `pdf_char_count`) and persists checkpoints to SQLite.

## 1. Check GPU & Environment

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Install Required Dependencies

In [ ]:
# Core inference packages
!pip install -q pyarrow pandas pymupdf pillow torch torchvision transformers accelerate bitsandbytes
# Extra dependencies for baidu/Unlimited-OCR (if running on A100)
!pip install -q einops addict easydict

## 4. Run Batch OCR Script

Adjust `--input-dir` and `--output-dir` to point to your Google Drive directory.

In [ ]:
!python3 scripts/semanticscholar_pipeline/colab_ocr_batch.py \
    --input-dir /content/drive/MyDrive/SemanticScholar/Parquet \
    --output-dir /content/drive/MyDrive/SemanticScholar/OCR_Enriched \
    --model auto

## 5. Inspect Checkpoint & Success Metrics

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('ocr_progress.sqlite')
stats_df = pd.read_sql_query('SELECT status, model, count(*) as count, avg(char_count) as avg_chars FROM completed_papers GROUP BY status, model', conn)
display(stats_df)